# Bayesian inference foundations — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Posterior belief is prior belief multiplied by observed evidence
Bayesian inference updates a whole distribution, not a single best guess. These five examples use the Beta-Bernoulli model to show conjugate updating, the predictive mean, posterior uncertainty, likelihood influence, and model evidence.

In [ ]:
# 1) Beta-Bernoulli conjugate update: alpha and beta are pseudo-counts
x=np.linspace(0.001,0.999,400); a0,b0=2,3; s,f=7,3; a1,b1=a0+s,b0+f
plt.figure(figsize=(6,3)); plt.plot(x,beta_pdf_grid(x,a0,b0),label='prior Beta(2,3)'); plt.plot(x,beta_pdf_grid(x,a1,b1),label='posterior Beta(9,6)')
plt.legend(); plt.xlabel('theta'); plt.ylabel('density'); plt.title('prior counts + data counts')
assert (a1,b1)==(9,6)

In [ ]:
# 2) posterior predictive probability is the posterior mean
x=np.linspace(0.001,0.999,400); a,b=9,6; mean=a/(a+b)
plt.figure(figsize=(6,3)); plt.plot(x,beta_pdf_grid(x,a,b)); plt.axvline(mean,c='r',ls='--'); plt.title(f'predict next success = {mean:.3f}')
assert abs(mean-0.6)<1e-12

In [ ]:
# 3) uncertainty shrinks after data: Beta variance before vs after
var0=a0*b0/((a0+b0)**2*(a0+b0+1)); var1=a1*b1/((a1+b1)**2*(a1+b1+1))
plt.figure(figsize=(6,3)); plt.bar(['prior','posterior'],[var0,var1]); plt.ylabel('variance'); plt.title('data narrows belief')
assert abs(var0-0.04)<1e-12 and abs(var1-0.015)<1e-12

In [ ]:
# 4) prior odds times likelihood ratio gives posterior odds
prior_odds=(0.7**(a0-1)*(0.3**(b0-1)))/(0.4**(a0-1)*(0.6**(b0-1)))
like_ratio=(0.7**s*0.3**f)/(0.4**s*0.6**f); post_odds=prior_odds*like_ratio
plt.figure(figsize=(6,3)); plt.bar(['prior odds','likelihood ratio','posterior odds'],[prior_odds,like_ratio,post_odds]); plt.yscale('log'); plt.title('posterior odds = prior odds × likelihood ratio')
assert abs(post_odds-2.7488713264465314)<1e-9

In [ ]:
# 5) model evidence is the average likelihood under the prior
from math import gamma, comb
prior_pred=comb(10,7)*gamma(a0+s)*gamma(b0+f)*gamma(a0+b0)/(gamma(a0)*gamma(b0)*gamma(a0+b0+10))
plt.figure(figsize=(6,3)); plt.bar(['P(7 successes in 10)'],[prior_pred]); plt.ylim(0,0.3); plt.title(f'evidence = {prior_pred:.3f}')
assert abs(prior_pred-0.07992007992007992)<1e-12